# Classification & Regression Metrics

**Companion lesson:** https://ml-viz.vercel.app/courses/model-evaluation/01-classification-metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (roc_curve, precision_recall_curve, auc,
                              roc_auc_score, average_precision_score)

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. Confusion Matrix

Generate synthetic binary predictions and visualize the confusion matrix with all four derived metrics.

In [ ]:
# Generate synthetic binary labels and scores
n = 500
y_true = (rng.uniform(0, 1, n) < 0.3).astype(int)  # 30% positive
# Scores: positives skewed high, negatives skewed low
scores = np.where(y_true == 1,
                  rng.beta(5, 2, n),   # positives: high scores
                  rng.beta(2, 5, n))   # negatives: low scores

# Apply threshold 0.5
y_pred = (scores >= 0.5).astype(int)

TP = np.sum((y_pred == 1) & (y_true == 1))
FP = np.sum((y_pred == 1) & (y_true == 0))
TN = np.sum((y_pred == 0) & (y_true == 0))
FN = np.sum((y_pred == 0) & (y_true == 1))

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)
accuracy  = (TP + TN) / n

# Plot confusion matrix
cm = np.array([[TN, FP], [FN, TP]])
labels = [['TN', 'FP'], ['FN', 'TP']]

fig, (ax_cm, ax_metrics) = plt.subplots(1, 2, figsize=(12, 4))

im = ax_cm.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{labels[i][j]}\n{cm[i,j]}',
                   ha='center', va='center', color='white', fontsize=14, fontweight='bold')
ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(['Pred Neg', 'Pred Pos'])
ax_cm.set_yticklabels(['Actual Neg', 'Actual Pos'])
ax_cm.set_title('Confusion Matrix', color='white')
plt.colorbar(im, ax=ax_cm)

metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1']
metrics_values = [accuracy, precision, recall, f1]
bar_colors     = [BRAND, TEAL, YELLOW, ROSE]
bars = ax_metrics.bar(metrics_names, metrics_values, color=bar_colors, edgecolor='none')
for bar, val in zip(bars, metrics_values):
    ax_metrics.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=11)
ax_metrics.set_ylim(0, 1.1)
ax_metrics.set_title('Derived Metrics', color='white')
ax_metrics.set_ylabel('Score')
ax_metrics.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print(f'TP={TP}, FP={FP}, TN={TN}, FN={FN}')

## 2. Precision-Recall Tradeoff

Precision and recall trade off as we vary the classification threshold.
The F1-maximizing threshold is not always 0.5.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    yp = (scores >= t).astype(int)
    tp = np.sum((yp == 1) & (y_true == 1))
    fp = np.sum((yp == 1) & (y_true == 0))
    fn = np.sum((yp == 0) & (y_true == 1))
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    precisions.append(p)
    recalls.append(r)
    f1s.append(2 * p * r / (p + r + 1e-9))

best_t_idx = np.argmax(f1s)
best_t     = thresholds[best_t_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precisions, color=TEAL,   linewidth=2, label='Precision')
ax.plot(thresholds, recalls,    color=YELLOW,  linewidth=2, label='Recall')
ax.plot(thresholds, f1s,        color=BRAND,   linewidth=2, label='F1', linestyle='--')
ax.axvline(best_t, color=ROSE, linestyle=':', linewidth=1.5,
           label=f'Best F1 threshold = {best_t:.2f}')
ax.set_xlabel('Classification threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff vs. Threshold', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best threshold: {best_t:.2f} → F1={max(f1s):.3f}')

## 3. ROC vs PR Curves on Imbalanced Data

With only 5% positive rate, the ROC curves for two classifiers look similar,
but the PR curves reveal that Classifier B is much worse in practice.

In [ ]:
rng2 = np.random.default_rng(7)
n2 = 2000
y2 = (rng2.uniform(0, 1, n2) < 0.05).astype(int)  # 5% positive

# Good classifier
scores_a = np.where(y2 == 1, rng2.beta(8, 2, n2), rng2.beta(2, 8, n2))
# Worse classifier (less separation)
scores_b = np.where(y2 == 1, rng2.beta(4, 3, n2), rng2.beta(3, 4, n2))

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))

for scores, label, color in [(scores_a, 'Classifier A', TEAL), (scores_b, 'Classifier B', ROSE)]:
    fpr, tpr, _ = roc_curve(y2, scores)
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={roc_auc:.3f})')

    prec, rec, _ = precision_recall_curve(y2, scores)
    pr_auc = auc(rec, prec)
    ax_pr.plot(rec, prec, color=color, linewidth=2, label=f'{label} (AUC={pr_auc:.3f})')

ax_roc.plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random (AUC=0.5)')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves (looks similar!)', color='white')
ax_roc.legend(); ax_roc.grid(True, alpha=0.3)

ax_pr.axhline(y=0.05, color='gray', linestyle='--', alpha=0.5, label='No-skill (prevalence=5%)')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('PR Curves (difference is clear!)', color='white')
ax_pr.legend(); ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Regression Metrics

MAE, MSE, RMSE, and R² on a noisy prediction problem. See how outliers affect MSE disproportionately.

In [ ]:
rng3 = np.random.default_rng(13)
x_reg = np.linspace(0, 10, 100)
y_true_reg = np.sin(x_reg) + 0.5 * x_reg

# Add noise and a few outliers
noise = rng3.normal(0, 0.3, 100)
outlier_idx = [20, 50, 80]
noise[outlier_idx] = rng3.normal(0, 3, 3)  # large outlier errors
y_pred_reg = y_true_reg + noise

residuals = y_pred_reg - y_true_reg
mae  = np.mean(np.abs(residuals))
mse  = np.mean(residuals**2)
rmse = np.sqrt(mse)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_true_reg - y_true_reg.mean())**2)
r2 = 1 - ss_res / ss_tot

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(x_reg, y_true_reg, color=TEAL, linewidth=2, label='True')
ax1.scatter(x_reg, y_pred_reg, color=BRAND, s=15, alpha=0.7, label='Predicted')
for idx in outlier_idx:
    ax1.annotate('outlier', (x_reg[idx], y_pred_reg[idx]), fontsize=8,
                 color=ROSE, xytext=(x_reg[idx]+0.3, y_pred_reg[idx]+0.3))
ax1.set_title(f'Predictions  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}', color='white')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.hist(residuals, bins=25, color=BRAND, alpha=0.7, edgecolor='none')
ax2.axvline(0, color=TEAL, linewidth=1.5, linestyle='--')
ax2.set_xlabel('Residual (y_pred - y_true)')
ax2.set_title('Residual distribution (outliers visible as long tail)', color='white')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'MAE  = {mae:.4f}  (average absolute error)')
print(f'MSE  = {mse:.4f}  (average squared error — outliers dominate)')
print(f'RMSE = {rmse:.4f}  (same units as y)')
print(f'R²   = {r2:.4f}  (fraction of variance explained)')

---
## ✏️ Your turn

Each exercise has a stub to fill in, an assert cell that prints `✅` when correct, and a solution in `<details>`.

### Exercise 1 — `compute_metrics(y_true, y_pred)`

Compute precision, recall, F1, and accuracy from scratch (no sklearn).

Return a dict `{'precision': ..., 'recall': ..., 'f1': ..., 'accuracy': ...}`.

Formulas:
- $\text{Precision} = TP / (TP + FP)$
- $\text{Recall} = TP / (TP + FN)$
- $F_1 = 2 \cdot P \cdot R / (P + R)$
- $\text{Accuracy} = (TP + TN) / N$

In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Compute classification metrics from binary labels.
    Returns dict with precision, recall, f1, accuracy.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    # TODO(you): compute TP, FP, TN, FN then derive the four metrics
    TP = ...
    FP = ...
    TN = ...
    FN = ...
    precision = ...
    recall    = ...
    f1        = ...
    accuracy  = ...
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

In [ ]:
from sklearn import metrics as sk_metrics

yt = rng.integers(0, 2, 200)
yp = rng.integers(0, 2, 200)

result = compute_metrics(yt, yp)
assert result is not None, "compute_metrics returned None"
assert abs(result['precision'] - sk_metrics.precision_score(yt, yp, zero_division=0)) < 1e-9, \
    f"Precision mismatch: {result['precision']:.4f} vs {sk_metrics.precision_score(yt, yp, zero_division=0):.4f}"
assert abs(result['recall'] - sk_metrics.recall_score(yt, yp, zero_division=0)) < 1e-9, \
    f"Recall mismatch"
assert abs(result['f1'] - sk_metrics.f1_score(yt, yp, zero_division=0)) < 1e-9, \
    f"F1 mismatch"
assert abs(result['accuracy'] - sk_metrics.accuracy_score(yt, yp)) < 1e-9, \
    f"Accuracy mismatch"
print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    TP = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / len(y_true)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}
```

</details>

### Exercise 2 — `roc_auc_from_scratch(y_true, y_scores)`

Compute AUC-ROC from scratch without sklearn:
1. Sort predictions by score descending
2. Walk the sorted list, accumulating TPR and FPR at each threshold
3. Compute the area under this curve via the trapezoid rule

**Hint:** at each position, count cumulative TP and FP.

In [ ]:
def roc_auc_from_scratch(y_true, y_scores):
    """
    Compute ROC AUC without sklearn.
    Returns scalar AUC in [0, 1].
    """
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    # TODO(you): sort by score descending, trace TPR vs FPR, compute trapezoid area
    # Hint: sort_idx = np.argsort(y_scores)[::-1]
    #       then iterate accumulating TP and FP counts
    #       TPR = cumulative_TP / total_positives
    #       FPR = cumulative_FP / total_negatives
    return ...

In [ ]:
# Test on the scores from earlier
my_auc_a = roc_auc_from_scratch(y2, scores_a)
my_auc_b = roc_auc_from_scratch(y2, scores_b)
sk_auc_a = roc_auc_score(y2, scores_a)
sk_auc_b = roc_auc_score(y2, scores_b)

assert my_auc_a is not None, "roc_auc_from_scratch returned None"
assert abs(my_auc_a - sk_auc_a) < 0.01, \
    f"Classifier A: expected {sk_auc_a:.4f}, got {my_auc_a:.4f}"
assert abs(my_auc_b - sk_auc_b) < 0.01, \
    f"Classifier B: expected {sk_auc_b:.4f}, got {my_auc_b:.4f}"
assert my_auc_a > my_auc_b, "Classifier A should have higher AUC"
print(f"AUC A: yours={my_auc_a:.4f}, sklearn={sk_auc_a:.4f}")
print(f"AUC B: yours={my_auc_b:.4f}, sklearn={sk_auc_b:.4f}")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def roc_auc_from_scratch(y_true, y_scores):
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    sort_idx = np.argsort(y_scores)[::-1]
    y_sorted = y_true[sort_idx]
    total_pos = y_true.sum()
    total_neg = len(y_true) - total_pos
    tprs = [0.0]
    fprs = [0.0]
    tp, fp = 0, 0
    for label in y_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1
        tprs.append(tp / total_pos)
        fprs.append(fp / total_neg)
    return float(np.trapz(tprs, fprs))
```

</details>